# LeRobot dataset inspection (matplotlib) — optional Rerun `.rrd`

**Default**: plot **camera video** (HTML animation over the episode, length from data / `meta/episodes` `length`, playback speed from `info.json` **fps**) plus **`observation.state` / `action` vs time** with a moving cursor, and **metadata**. Set `MPL_ANIMATE_CAMERAS=False` for a single still at `MPL_LOCAL_FRAME`. Increase `MPL_VIDEO_MAX_FRAMES` if you need more than ~200 sampled frames (decoding is slow).

**After** README §4 conversion, set `DATASET_ROOT` and episode selection in the config cell.

## Episode selection

- **Multi-task preview (`all_task_*`)**: set `EPISODES_PER_TASK = 3` to take the **first 3 episodes per distinct** `tasks` label in `meta/episodes` (in `episode_index` order). Overrides `EXPORT_ALL_EPISODES`, `TASK_FILTER`, `START_EPISODE`, `NUM_EPISODES`, and `EPISODE_INDICES`. Use `0` to disable.
- **Entire dataset**: set `EXPORT_ALL_EPISODES = True` (reads `total_episodes` from `meta/info.json`). Ignores `TASK_FILTER`, `START_EPISODE`, `NUM_EPISODES`, and `MAX_EPISODES` (and is skipped if `EPISODES_PER_TASK > 0`).
- **`all_task_*`**: use `TASK_FILTER` (substring / snake_case) or leave empty and use `START_EPISODE` + `NUM_EPISODES`.
- **`EPISODE_INDICES`**: if non-empty, overrides filter/range (unless `EXPORT_ALL_EPISODES` or `EPISODES_PER_TASK > 0`).
- **`MAX_EPISODES`**: `None` = no cap on how many indices you keep (e.g. all episodes matching `TASK_FILTER`). Use an integer to limit.
- **Matplotlib target**: `MPL_EPISODE_GLOBAL` or, if `MPL_USE_FIRST_RESOLVED_EPISODE=True`, the first entry in the resolved `episodes` list.

## Rerun over SSH (headless server, browser on your laptop)

Native `rerun a.rrd` needs **X11/Wayland**. For **web UI**, Rerun listens on **two** ports on the server (defaults):

- **9090** — HTTP web viewer page  
- **9876** — gRPC proxy the page’s JavaScript talks to (`localhost:9876` **in the browser**)

So you must forward **both** to your laptop, and use **free local ports** if 9090 is already in use (e.g. `Address already in use` on macOS).

**1) On the remote host** (where `.rrd` files live):

```bash
rerun --serve-web --web-viewer-port 9090 /path/to/a.rrd /path/to/b.rrd
```

**2) On your laptop** — pick unused locals, e.g. `19090` and `19876`:

```bash
ssh -N -L 19090:127.0.0.1:9090 -L 19876:127.0.0.1:9876 user@remote-host
```

**3) Open this URL** in your browser (adjust ports if you chose different locals):

`http://127.0.0.1:19090?url=rerun%2Bhttp%3A%2F%2F127.0.0.1%3A19876%2Fproxy`

Forwarding only 9090 breaks the viewer because the page still tries to reach **your laptop’s** `127.0.0.1:9876`, not the server’s.

`VIZ_MODE` / `EXPORT_RRD` below control `.rrd` export; matplotlib does not need Rerun. **RRD reuse**: with `RRD_FORCE_REGENERATE=False`, the RRD cell skips `lerobot_dataset_viz` when `local_<dataset>_episode_<N>.rrd` already exists under `output/` (only missing episodes are exported).

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO_ROOT = Path("/home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_based_robot_control").resolve()
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)

LEROBOT_SRC = REPO_ROOT / "external" / "lerobot" / "src"
VIZ_SCRIPT = LEROBOT_SRC / "lerobot" / "scripts" / "lerobot_dataset_viz.py"
assert VIZ_SCRIPT.is_file(), f"Missing {VIZ_SCRIPT}"

def _extend_path_front(key: str, path: Path) -> None:
    path = str(path)
    cur = os.environ.get(key, "")
    parts = cur.split(os.pathsep) if cur else []
    if path not in parts:
        os.environ[key] = path + (os.pathsep + cur if cur else "")

_extend_path_front("PYTHONPATH", LEROBOT_SRC)
for p in (REPO_ROOT, REPO_ROOT / "src"):
    ps = str(p)
    if ps not in sys.path:
        sys.path.insert(0, ps)

REPO_ROOT: /home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_based_robot_control


In [2]:
import json

# --- Dataset ---
DATASET_ROOT = REPO_ROOT / "datasets" / "lerobot_trial_3" / "all_task_eef"  # or all_task_eef

TASK_FILTER = ""  # ignored when EPISODES_PER_TASK > 0
START_EPISODE = 0
NUM_EPISODES = 2
EPISODE_INDICES: list[int] = []
# Cap task-filtered / explicit episode lists, and START+NUM range. None = no cap.
MAX_EPISODES = None
# True → all episodes 0..total_episodes-1 (from meta/info.json). Ignores TASK_FILTER, START_EPISODE, NUM_EPISODES, MAX_EPISODES.
EXPORT_ALL_EPISODES = False
# Multi-task merge (e.g. all_task_eef / all_task_joint): first N episodes per distinct `tasks` string in meta/episodes.
# If > 0, overrides EXPORT_ALL_EPISODES, TASK_FILTER, START_EPISODE, NUM_EPISODES, EPISODE_INDICES. MAX_EPISODES still caps the final list.
EPISODES_PER_TASK = 3

OUTPUT_DIR = None  # RRD output; default DATASET_ROOT / "output"
VIDEO_BACKEND = "pyav"

# --- Visualization mode ---
# "matplotlib" | "rrd_export" | "both"
VIZ_MODE = "rrd_export"

# Matplotlib: which global episode_index to plot
MPL_USE_FIRST_RESOLVED_EPISODE = True
MPL_EPISODE_GLOBAL = 0
# Camera row: animate full episode (length from data/meta); False = single still at MPL_LOCAL_FRAME
MPL_ANIMATE_CAMERAS = True
MPL_LOCAL_FRAME = 0  # used only when MPL_ANIMATE_CAMERAS is False
# Cap how many timesteps to decode (evenly spaced in the range below) — full decode is slow
MPL_VIDEO_MAX_FRAMES = 200
# Local frame indices inside the episode (same as parquet row order for that episode)
MPL_VIDEO_LOCAL_START = 0
MPL_VIDEO_LOCAL_END = None  # inclusive; None = last frame

# RRD export (slow; uses lerobot_dataset_viz.py subprocess per episode)
EXPORT_RRD = False
# If False, skip lerobot_dataset_viz when the expected .rrd for that episode already exists
RRD_FORCE_REGENERATE = False
BATCH_SIZE = 16
NUM_WORKERS = 0
TOLERANCE_S = 1e-4

# Native GUI rerun (needs DISPLAY/Wayland — usually fails on pure SSH)
OPEN_RERUN_NATIVE = False

print("DATASET_ROOT:", DATASET_ROOT)
print("VIZ_MODE:", VIZ_MODE, "EXPORT_RRD:", EXPORT_RRD, "RRD_FORCE_REGENERATE:", RRD_FORCE_REGENERATE)
print("TASK_FILTER:", repr(TASK_FILTER))
print(
    "EPISODES_PER_TASK:", EPISODES_PER_TASK,
    "EXPORT_ALL_EPISODES:", EXPORT_ALL_EPISODES,
    "MAX_EPISODES:", MAX_EPISODES,
)

DATASET_ROOT: /home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_based_robot_control/datasets/lerobot_trial_3/all_task_eef
VIZ_MODE: rrd_export EXPORT_RRD: False RRD_FORCE_REGENERATE: False
TASK_FILTER: ''
EPISODES_PER_TASK: 3 EXPORT_ALL_EPISODES: False MAX_EPISODES: None


In [3]:
import glob
import json
from collections import Counter

import numpy as np
import pandas as pd


def _cap_episodes(idxs: list[int], max_episodes: int | None) -> list[int]:
    if max_episodes is None:
        return idxs
    return idxs[: int(max_episodes)]


def _tasks_cell_to_str(cell) -> str:
    if cell is None:
        return ""
    if isinstance(cell, np.ndarray):
        cell = cell.tolist()
    if isinstance(cell, (list, tuple)):
        cell = cell[0] if len(cell) else ""
    return str(cell).strip()


def _needles_for_task_filter(task_filter: str) -> list[str]:
    s = task_filter.strip()
    if not s:
        return []
    low = s.lower()
    out = [low]
    if "_" in s and " " not in s and s.isascii():
        out.append(low.replace("_", " "))
    return list(dict.fromkeys(out))


def load_episodes_table(dataset_root: Path) -> pd.DataFrame:
    pattern = str(dataset_root / "meta" / "episodes" / "**" / "*.parquet")
    paths = sorted(glob.glob(pattern, recursive=True))
    if not paths:
        raise FileNotFoundError(f"No episode parquet under {dataset_root}/meta/episodes")
    return pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)


def episode_indices_per_task_sample(dataset_root: Path, per_task: int) -> list[int]:
    """First `per_task` rows per distinct `tasks` cell, in global episode_index order."""
    if per_task <= 0:
        raise ValueError("per_task must be > 0")
    ep_df = load_episodes_table(dataset_root)
    if "tasks" not in ep_df.columns:
        raise ValueError("EPISODES_PER_TASK requires meta/episodes 'tasks' column")
    ep_df = ep_df.sort_values("episode_index", kind="mergesort")
    counts: dict[str, int] = {}
    picked: list[int] = []
    for _, row in ep_df.iterrows():
        key = _tasks_cell_to_str(row["tasks"]) or "(no task text)"
        n = counts.get(key, 0)
        if n >= per_task:
            continue
        counts[key] = n + 1
        picked.append(int(row["episode_index"]))
    return sorted(picked)


def episode_indices_to_export(
    dataset_root: Path,
    task_filter: str,
    start_episode: int,
    num_episodes: int,
    explicit: list[int],
    max_episodes: int | None,
    export_all: bool,
    episodes_per_task: int,
) -> list[int]:
    if episodes_per_task > 0:
        return _cap_episodes(
            episode_indices_per_task_sample(dataset_root, episodes_per_task),
            max_episodes,
        )
    if export_all:
        with open(dataset_root / "meta" / "info.json") as f:
            total = int(json.load(f)["total_episodes"])
        return list(range(total))
    if explicit:
        return _cap_episodes(sorted(set(int(x) for x in explicit)), max_episodes)
    if not task_filter.strip():
        r = list(range(start_episode, start_episode + num_episodes))
        return _cap_episodes(r, max_episodes)
    ep_df = load_episodes_table(dataset_root)
    if "tasks" not in ep_df.columns:
        raise ValueError("TASK_FILTER set but episodes parquet has no 'tasks' column")
    needles = _needles_for_task_filter(task_filter)
    hits: list[int] = []
    for _, row in ep_df.iterrows():
        text = _tasks_cell_to_str(row["tasks"]).lower()
        if any(n for n in needles if n in text):
            hits.append(int(row["episode_index"]))
    hits = sorted(set(hits))
    if not hits:
        raise ValueError(
            f"No episodes matched TASK_FILTER={task_filter!r} (needles={needles!r}). "
            "Inspect meta/episodes 'tasks' or clear TASK_FILTER."
        )
    return _cap_episodes(hits, max_episodes)


dataset_root = Path(DATASET_ROOT)
if not (dataset_root / "meta" / "info.json").is_file():
    raise FileNotFoundError(f"Missing meta/info.json under {dataset_root}")

episodes = episode_indices_to_export(
    dataset_root,
    TASK_FILTER,
    START_EPISODE,
    NUM_EPISODES,
    EPISODE_INDICES,
    MAX_EPISODES,
    EXPORT_ALL_EPISODES,
    EPISODES_PER_TASK,
)
print("Resolved episode_index list:", episodes)
print("Count:", len(episodes))
if EPISODES_PER_TASK > 0:
    edf = load_episodes_table(dataset_root)
    c = Counter()
    for ei in episodes:
        r = edf.loc[edf["episode_index"] == ei, "tasks"]
        if len(r) == 0:
            continue
        lab = _tasks_cell_to_str(r.iloc[0])
        short = lab[:72] + ("…" if len(lab) > 72 else "")
        c[short] += 1
    print("Episodes per task label (truncated):", dict(c))

Resolved episode_index list: [0, 1, 2, 3, 4, 5, 6, 23, 50, 51, 52, 53, 54, 55, 72, 73, 102, 103, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 166, 167, 169, 170, 171, 172, 177, 178, 179, 181, 184, 189, 194, 195, 196, 197, 198, 199, 294, 295, 296, 297, 298, 299]
Count: 62
Episodes per task label (truncated): {'This prompt describes a robotic manipulation task using scene graphs. It…': 62}


In [4]:
if VIZ_MODE not in ("matplotlib", "both"):
    print("Skip matplotlib (VIZ_MODE=", VIZ_MODE, ")")
else:
    import matplotlib.pyplot as plt
    import torch
    from lerobot.datasets.lerobot_dataset import LeRobotDataset

    def chw_float_to_hwc_uint8(t: torch.Tensor) -> np.ndarray:
        assert t.dtype == torch.float32 and t.ndim == 3
        c, h, w = t.shape
        assert c < h and c < w
        return (t * 255).clamp(0, 255).to(torch.uint8).permute(1, 2, 0).cpu().numpy()

    ep_plot = episodes[0] if MPL_USE_FIRST_RESOLVED_EPISODE else MPL_EPISODE_GLOBAL
    if ep_plot not in set(episodes):
        print(f"Note: MPL_EPISODE_GLOBAL={ep_plot} not in resolved list {episodes}; plotting anyway.")

    with open(dataset_root / "meta" / "info.json") as f:
        info = json.load(f)
    feats = info.get("features", {})
    st_names = feats.get("observation.state", {}).get("names") or [f"s{i}" for i in range(8)]
    ac_names = feats.get("action", {}).get("names") or [f"a{i}" for i in range(8)]

    data_paths = sorted(glob.glob(str(dataset_root / "data" / "**" / "*.parquet"), recursive=True))
    if not data_paths:
        raise FileNotFoundError("No data parquet under dataset_root/data")
    data_df = pd.concat([pd.read_parquet(p) for p in data_paths], ignore_index=True)
    sub = data_df[data_df["episode_index"] == ep_plot].sort_values("index")
    if sub.empty:
        raise ValueError(f"No rows for episode_index={ep_plot}")

    states = np.stack(sub["observation.state"].values)
    actions = np.stack(sub["action"].values)
    t_axis = np.arange(len(sub))
    task_idx = int(sub["task_index"].iloc[0])
    tasks_path = dataset_root / "meta" / "tasks.parquet"
    if tasks_path.is_file():
        tdf = pd.read_parquet(tasks_path)
        if 0 <= task_idx < len(tdf):
            task_text = str(tdf.iloc[task_idx].name)
        else:
            task_text = f"task_index={task_idx} (out of range)"
    else:
        task_text = f"task_index={task_idx}"

    meta_eps = load_episodes_table(dataset_root)
    ep_row = meta_eps[meta_eps["episode_index"] == ep_plot]
    ep_tasks = ""
    if not ep_row.empty and "tasks" in ep_row.columns:
        ep_tasks = _tasks_cell_to_str(ep_row.iloc[0]["tasks"])

    print("--- Metadata ---")
    print("episode_index:", ep_plot, "  frames:", len(sub))
    print("task (tasks.parquet):", task_text)
    print("episode tasks cell:", ep_tasks or "(n/a)")
    fps_meta = float(info.get("fps") or 10)
    print("fps (meta):", fps_meta)
    T = len(sub)
    if not ep_row.empty and "length" in ep_row.columns:
        T_meta = int(ep_row.iloc[0]["length"])
        if T_meta != T:
            print(f"Note: meta/episodes length={T_meta} vs data rows={T} (using data rows for time axis)")

    repo_id = f"local/{dataset_root.name}"
    ds = LeRobotDataset(
        repo_id=repo_id,
        root=dataset_root,
        episodes=[ep_plot],
        download_videos=True,
        video_backend=VIDEO_BACKEND,
    )
    T_ds = len(ds)
    if T_ds != T:
        print(f"Note: LeRobotDataset len={T_ds} vs parquet T={T} — animation uses min(T_ds,T) local indices")
    T_eff = min(T_ds, T)
    cam_keys = sorted(k for k in ds[0] if k.startswith("observation.images."))

    n_st, n_ac = states.shape[1], actions.shape[1]
    fig_h = 3 + 2 * int(np.ceil(n_st / 4)) + 2 * int(np.ceil(n_ac / 4))
    fig = plt.figure(figsize=(14, fig_h))
    gs = fig.add_gridspec(3, 2, height_ratios=[1.2, 1, 1], hspace=0.35, wspace=0.2)

    ims: list = []
    anim_local_idx: list[int] = []
    if MPL_ANIMATE_CAMERAS and T_eff > 0:
        _end = T_eff - 1 if MPL_VIDEO_LOCAL_END is None else min(T_eff - 1, int(MPL_VIDEO_LOCAL_END))
        _start = max(0, min(int(MPL_VIDEO_LOCAL_START), _end))
        cand = list(range(_start, _end + 1))
        cap = max(1, int(MPL_VIDEO_MAX_FRAMES))
        if len(cand) <= cap:
            anim_local_idx = cand
        else:
            pick = np.unique(np.round(np.linspace(0, len(cand) - 1, cap)).astype(int)).tolist()
            anim_local_idx = [cand[i] for i in pick]
        print(
            f"Camera video: {len(anim_local_idx)} frames sampled in local range [{_start}, {_end}] "
            f"(episode len {T_eff}; ~{fps_meta:.1f} fps → {1000 / fps_meta:.0f} ms/frame)"
        )
        print("Decoding frames from mp4 (slow for long episodes)...")
        vid_rgbs: list[list[np.ndarray]] = []
        for li in anim_local_idx:
            it = ds[li]
            vid_rgbs.append([chw_float_to_hwc_uint8(it[ck]) for ck in cam_keys[:2]])
        for i, ck in enumerate(cam_keys[:2]):
            ax = fig.add_subplot(gs[0, i])
            im = ax.imshow(vid_rgbs[0][i])
            ax.set_title(f"{ck.replace('observation.images.', '')}  (ep {ep_plot}, video)")
            ax.axis("off")
            ims.append(im)
        if len(cam_keys) > 2:
            print("(More cameras in data; animating first two.)")
    else:
        ft = min(MPL_LOCAL_FRAME, T_ds - 1)
        if MPL_LOCAL_FRAME >= T_ds:
            print(f"MPL_LOCAL_FRAME capped to {ft} (episode length {T_ds})")
        item = ds[ft]
        for i, ck in enumerate(cam_keys[:2]):
            ax = fig.add_subplot(gs[0, i])
            ax.imshow(chw_float_to_hwc_uint8(item[ck]))
            ax.set_title(f"{ck.replace('observation.images.', '')}  (ep {ep_plot}, local t={ft})")
            ax.axis("off")
        if len(cam_keys) > 2:
            print("(More cameras in data; showing first two.)")
        vid_rgbs = []

    ax_s = fig.add_subplot(gs[1, :])
    for j in range(n_st):
        nm = st_names[j] if j < len(st_names) else f"{j}"
        ax_s.plot(t_axis, states[:, j], label=nm, alpha=0.85)
    ax_s.set_title("observation.state")
    ax_s.set_xlabel("frame (within episode)")
    ax_s.legend(loc="upper right", fontsize=7, ncol=2)
    ax_s.grid(True, alpha=0.3)

    ax_a = fig.add_subplot(gs[2, :])
    for j in range(n_ac):
        nm = ac_names[j] if j < len(ac_names) else f"{j}"
        ax_a.plot(t_axis, actions[:, j], label=nm, alpha=0.85)
    ax_a.set_title("action")
    ax_a.set_xlabel("frame (within episode)")
    ax_a.legend(loc="upper right", fontsize=7, ncol=2)
    ax_a.grid(True, alpha=0.3)

    fig.suptitle(f"{task_text[:120]}{'…' if len(task_text) > 120 else ''}", fontsize=10)

    if MPL_ANIMATE_CAMERAS and vid_rgbs:
        from matplotlib import animation
        from IPython.display import HTML, display

        vline_s = ax_s.axvline(anim_local_idx[0], color="0.15", lw=1.5, alpha=0.65)
        vline_a = ax_a.axvline(anim_local_idx[0], color="0.15", lw=1.5, alpha=0.65)
        interval_ms = max(1, int(1000 / fps_meta))

        def _update(_j: int):
            x = float(anim_local_idx[_j])
            for _im, _rgb in zip(ims, vid_rgbs[_j]):
                _im.set_data(_rgb)
            vline_s.set_xdata([x, x])
            vline_a.set_xdata([x, x])
            return ims + [vline_s, vline_a]

        anim = animation.FuncAnimation(
            fig, _update, frames=len(vid_rgbs), interval=interval_ms, blit=False
        )
        plt.close(fig)
        display(HTML(anim.to_jshtml(default_mode="loop")))
    else:
        plt.show()

Skip matplotlib (VIZ_MODE= rrd_export )


In [5]:
_do_rrd = EXPORT_RRD or VIZ_MODE in ("rrd_export", "both")
if not _do_rrd:
    print("Skip RRD export (EXPORT_RRD=False and VIZ_MODE not rrd_export/both).")
else:
    import shutil
    import subprocess

    out = Path(OUTPUT_DIR) if OUTPUT_DIR else dataset_root / "output"
    out.mkdir(parents=True, exist_ok=True)
    dataset_id = dataset_root.name
    repo_id = f"local/{dataset_id}"
    env = os.environ.copy()
    env["PYTHONPATH"] = str(LEROBOT_SRC) + os.pathsep + env.get("PYTHONPATH", "")
    env.setdefault("CUDA_VISIBLE_DEVICES", "")

    def _expected_rrd(ep: int) -> Path:
        return out / f"{repo_id.replace('/', '_')}_episode_{ep}.rrd"

    rrd_paths: list[Path] = []
    for ep in episodes:
        rrd = _expected_rrd(ep)
        if rrd.is_file() and not RRD_FORCE_REGENERATE:
            print(f"[rrd] skip episode {ep} (already exists: {rrd})")
            rrd_paths.append(rrd)
            continue
        cmd = [
            sys.executable,
            str(VIZ_SCRIPT),
            "--repo-id",
            repo_id,
            "--root",
            str(dataset_root),
            "--episode-index",
            str(ep),
            "--save",
            "1",
            "--output-dir",
            str(out),
            "--batch-size",
            str(BATCH_SIZE),
            "--num-workers",
            str(NUM_WORKERS),
            "--tolerance-s",
            str(TOLERANCE_S),
            "--video-backend",
            VIDEO_BACKEND,
        ]
        print("\n[rrd export] episode", ep)
        subprocess.run(cmd, env=env, check=True)
        if not rrd.is_file():
            raise FileNotFoundError(f"Missing {rrd} after export")
        rrd_paths.append(rrd)

    print("\nRRD files:")
    for p in rrd_paths:
        print(" ", p)

    paths_s = " ".join(str(p) for p in rrd_paths)
    print("\n--- Native GUI (needs DISPLAY/Wayland on this machine) ---")
    print("rerun", paths_s)
    print("\n--- Headless / SSH: forward BOTH web (9090) and gRPC (9876) ---")
    print("On this host:")
    print(f"  rerun --serve-web --web-viewer-port 9090 {paths_s}")
    print("On laptop (use free local ports if 9090 is busy, e.g. 19090 + 19876):")
    print("  ssh -N -L 19090:127.0.0.1:9090 -L 19876:127.0.0.1:9876 ngocbach@10.218.100.43")
    from urllib.parse import quote

    lw, lg = 19090, 19876
    browser_url = f"http://127.0.0.1:{lw}?url={quote(f'rerun+http://127.0.0.1:{lg}/proxy', safe='')}"
    print("  # then open:")
    print(" ", browser_url)

    rr = shutil.which("rerun")
    if OPEN_RERUN_NATIVE and rrd_paths:
        if not rr:
            print("rerun not on PATH; use commands above.")
        else:
            subprocess.run([rr, *[str(p) for p in rrd_paths]], check=False)


[rrd export] episode 0


INFO 2026-03-28 11:58:58 aset_viz.py:364 Loading dataset
INFO 2026-03-28 11:58:58 aset_viz.py:140 Loading dataloader
INFO 2026-03-28 11:58:58 aset_viz.py:147 Starting Rerun
INFO 2026-03-28 11:58:58 aset_viz.py:165 Logging to Rerun
  0%|          | 0/7 [00:00<?, ?it/s]/home/ngocbach@ad.asu.edu/anaconda3/envs/rlbench_gui/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
100%|██████████| 7/7 [00:02<00:00,  2.72it/s]



[rrd export] episode 1


INFO 2026-03-28 11:59:03 aset_viz.py:364 Loading dataset
INFO 2026-03-28 11:59:03 aset_viz.py:140 Loading dataloader
INFO 2026-03-28 11:59:03 aset_viz.py:147 Starting Rerun
INFO 2026-03-28 11:59:03 aset_viz.py:165 Logging to Rerun
  0%|          | 0/5 [00:00<?, ?it/s]/home/ngocbach@ad.asu.edu/anaconda3/envs/rlbench_gui/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
100%|██████████| 5/5 [00:01<00:00,  4.10it/s]



[rrd export] episode 2


...


[rrd export] episode 298


INFO 2026-03-28 12:02:54 aset_viz.py:364 Loading dataset
INFO 2026-03-28 12:02:54 aset_viz.py:140 Loading dataloader
INFO 2026-03-28 12:02:54 aset_viz.py:147 Starting Rerun
INFO 2026-03-28 12:02:54 aset_viz.py:165 Logging to Rerun
  0%|          | 0/5 [00:00<?, ?it/s]/home/ngocbach@ad.asu.edu/anaconda3/envs/rlbench_gui/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
100%|██████████| 5/5 [00:01<00:00,  3.60it/s]



[rrd export] episode 299


INFO 2026-03-28 12:02:57 aset_viz.py:364 Loading dataset
INFO 2026-03-28 12:02:57 aset_viz.py:140 Loading dataloader
INFO 2026-03-28 12:02:57 aset_viz.py:147 Starting Rerun
INFO 2026-03-28 12:02:58 aset_viz.py:165 Logging to Rerun
  0%|          | 0/5 [00:00<?, ?it/s]/home/ngocbach@ad.asu.edu/anaconda3/envs/rlbench_gui/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
100%|██████████| 5/5 [00:01<00:00,  4.75it/s]



RRD files:
  /home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_based_robot_control/datasets/lerobot_trial_3/all_task_eef/output/local_all_task_eef_episode_0.rrd
  /home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_based_robot_control/datasets/lerobot_trial_3/all_task_eef/output/local_all_task_eef_episode_1.rrd
  /home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_based_robot_control/datasets/lerobot_trial_3/all_task_eef/output/local_all_task_eef_episode_2.rrd
  /home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_based_robot_control/datasets/lerobot_trial_3/all_task_eef/output/local_all_task_eef_episode_3.rrd
  /home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_based_robot_control/datasets/lerobot_trial_3/all_task_eef/output/local_all_task_eef_episode_4.rrd
  /home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_based_robot_control/datasets/lerobot_trial_3/all_task_eef/output/local_all_task_eef_episode_5.rrd
  /home/ngocbach@ad.asu.edu/Desktop/cross_model_learning_b